<a href="https://colab.research.google.com/github/rajajisai/leet-torch-submissions/blob/main/v3/classical-ml/softmax/softmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implement Softmax from Scratch

**Difficulty**: 🟢 Easy

**Companies**: Apple, Meta, Google, Amazon, General FAANG

---

### Problem Statement

The softmax function is one of the most fundamental operations in deep learning. It converts a vector of raw scores (logits) into a probability distribution. A naive implementation can suffer from **numerical overflow** when input values are large.

Your task is to implement a **numerically stable** softmax function from scratch using only PyTorch tensor operations, and validate it against `torch.nn.functional.softmax`.

---

### Requirements

1. **Numerical Stability** — Subtract the maximum value along the target dimension before exponentiating to prevent overflow.
2. **Dimension Support** — The function must accept a `dim` argument (default `-1`) to specify which dimension to apply softmax over.
3. **Correctness** — Output must match `torch.nn.functional.softmax` within floating-point tolerance.
4. **Properties** — Output values must be in `[0, 1]` and sum to `1` along the specified dimension.

---

### Constraints

- ✅ Use only PyTorch tensor operations (no loops over elements).
- ✅ Must handle 1D, 2D, and higher-dimensional tensors.
- ✅ Must be numerically stable for large input values (e.g., `[1000, 2000, 3000]`).
- ❌ Do **not** use `torch.nn.functional.softmax` or `torch.softmax` in your implementation.

---

<details>
  <summary>💡 Hint</summary>

  - The trick to numerical stability: `softmax(x) == softmax(x - max(x))`. Subtracting the max does not change the result but prevents `exp()` from overflowing.
  - Use `x.max(dim=dim, keepdim=True).values` to get the max while preserving dimensions for broadcasting.
  - Remember to use `keepdim=True` in both `max()` and `sum()` so the shapes broadcast correctly.

</details>

---

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# Test data
torch.manual_seed(42)

# 1D input
x_1d = torch.randn(5)

# 2D input (batch of logits)
x_2d = torch.randn(3, 5)

# Large values to test numerical stability
x_large = torch.tensor([1000.0, 2000.0, 3000.0])

print("x_1d:", x_1d)
print("x_2d:", x_2d)
print("x_large:", x_large)

In [ ]:
def softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Compute numerically stable softmax along the specified dimension.

    Args:
        x (Tensor): Input tensor of any shape.
        dim (int): Dimension along which to compute softmax. Default: -1.

    Returns:
        Tensor: Softmax probabilities with the same shape as input.
    """
    # TODO: Step 1 - Subtract the max along `dim` for numerical stability (use keepdim=True)
    max_values = torch.max(x, dim=dim, keepdim=True).values
    x_shifted = x - max_values
    # TODO: Step 2 - Exponentiate the shifted values
    exp_values = torch.exp(x_shifted)
    # TODO: Step 3 - Normalize by dividing by the sum along `dim` (use keepdim=True)
    sum_values = torch.sum(exp_values, dim=dim, keepdim=True)
    return exp_values / sum_values
    ...

In [ ]:
# Validation
print("=== 1D Test ===")
out_1d = softmax(x_1d)
ref_1d = F.softmax(x_1d, dim=-1)
print("Custom:", out_1d)
print("PyTorch:", ref_1d)
assert torch.allclose(out_1d, ref_1d), "1D softmax mismatch!"
print("PASSED\n")

print("=== 2D Test ===")
out_2d = softmax(x_2d, dim=-1)
ref_2d = F.softmax(x_2d, dim=-1)
print("Custom:", out_2d)
print("PyTorch:", ref_2d)
assert torch.allclose(out_2d, ref_2d), "2D softmax mismatch!"
print("PASSED\n")

print("=== Numerical Stability Test ===")
out_large = softmax(x_large)
ref_large = F.softmax(x_large, dim=-1)
print("Custom:", out_large)
print("PyTorch:", ref_large)
assert torch.allclose(out_large, ref_large), "Numerical stability test failed!"
assert not torch.isnan(out_large).any(), "NaN detected in output!"
assert not torch.isinf(out_large).any(), "Inf detected in output!"
print("PASSED\n")

print("=== Sum-to-1 Test ===")
sums = softmax(x_2d, dim=-1).sum(dim=-1)
print("Row sums:", sums)
assert torch.allclose(sums, torch.ones_like(sums)), "Softmax rows do not sum to 1!"
print("PASSED\n")

print("All tests passed!")